# Financial Knowledge Graph — Chapters 01 & 04 Tutorial

**Audience:** Someone new to this codebase who wants to understand how a financial knowledge graph is built, stored in Neo4j, and analysed.

**What you will learn:**

| Chapter | Topic |
|---|---|
| `ch01_fin` | Platform setup — shared infrastructure, schema design, Neo4j constraints |
| `ch04_fin` | Multi-source financial graph — Exchanges, Instruments, Sectors, Ownership, Community Detection, PageRank |

---

## Prerequisites
- Neo4j 5.x running locally on `bolt://localhost:7687`
- Python venv activated
- Set `NEO4J_PASSWORD` in your environment (or update the constant below)

## Notebook structure
```
PART 1 — ch01_fin: Platform Foundation
  1.1  Why a Knowledge Graph for Finance?
  1.2  The Shared _platform Layer
  1.3  Data Model & Schema Design
  1.4  Neo4j Constraints & Indices
  1.5  Connecting to Neo4j
  1.6  Creating Sample Data (standalone demo)

PART 2 — ch04_fin: Building the Financial Graph
  2.1  Architecture Overview
  2.2  Step 1 — Exchange Nodes  (ISO 10383 MIC)
  2.3  Step 2 — Instrument Nodes (OpenFIGI)
  2.4  Step 3 — LISTED_ON Relationships
  2.5  Step 4 — Sector Mapping  (SEC SIC codes)
  2.6  Step 5 — Ownership Graph (SEC 13F filings)

PART 3 — Graph Exploration with Cypher
  3.1  Basic traversals
  3.2  Filtering and aggregation
  3.3  Shortest paths
  3.4  Community Detection (Louvain / connected-components)
  3.5  Centrality — PageRank
  3.6  Summary
```

---
# PART 1 — ch01_fin: Platform Foundation

## 1.1 Why a Knowledge Graph for Finance?

A **knowledge graph** (KG) stores information as *nodes* (entities) connected by *relationships* (edges). Finance is particularly well suited to a graph representation because:

1. **Ownership chains** — "Vanguard owns shares in Apple" is naturally a directed edge.
2. **Multi-hop risk propagation** — "Which companies share the same institutional shareholders?" requires path queries that SQL handles poorly.
3. **Heterogeneous sources** — GLEIF (legal entities), OpenFIGI (instruments), SEC EDGAR (filings), ISO (exchanges) can all be linked without a rigid relational schema.
4. **Ontology alignment** — SIC sector codes, FIBO financial ontology, and company hierarchies can be merged naturally.

### Core node types
```
LegalEntity   — a registered company (identified by LEI, CIK)
Instrument    — a tradeable security  (identified by FIGI, ISIN, ticker)
Exchange      — a trading venue       (identified by ISO 10383 MIC code)
OntologyClass — a taxonomy class      (SIC sector, FIBO concept)
Filing        — an SEC regulatory filing
```

### Core relationship types
```
(Instrument)  -[:LISTED_ON]->    (Exchange)
(Instrument)  -[:CLASSIFIED_AS]-> (OntologyClass)
(LegalEntity) -[:OWNS]->         (Instrument)
(LegalEntity) -[:PARENT_OF]->    (LegalEntity)
(LegalEntity) -[:FILED]->        (Filing)
```

## 1.2 The Shared `_platform` Layer

Every financial chapter shares a common infrastructure at `ChaptersFinancial/_platform/`:

```
_platform/
├── fin_importer_base.py   ← All importers extend this class
├── providers/
│   ├── graph.py           ← GraphProvider: thin wrapper around neo4j driver
│   ├── llm.py             ← LLMProvider: OpenAI / Ollama / mock
│   └── vector.py          ← VectorProvider: embeddings
├── schema/
│   └── constraints.cypher ← All CREATE CONSTRAINT / CREATE INDEX statements
├── config/
│   ├── schema_config.yaml
│   └── provider_config.yaml
├── obs/
│   ├── run_logger.py      ← JSONL audit trail per import run
│   └── cost_tracker.py    ← Token cost tracking
└── eval/
    ├── ner_eval.py, ned_eval.py, ml_eval.py, rag_eval.py
```

**Key design principle — `FinImporterBase` auto-injects:**
- `runId` (UUID) into every batch → trace any node to its exact import run
- `ingestedAt` timestamp
- Runs schema constraints on startup
- Provides `batch_store(cypher, iterator, size, desc)` with progress bar

## 1.3 Data Model & Schema Design

```
                    CLASSIFIED_AS
  (Instrument) ─────────────────────► (OntologyClass)
       │
       │ LISTED_ON
       ▼
  (Exchange)

  (LegalEntity) ──OWNS──► (Instrument)
  (LegalEntity) ──PARENT_OF──► (LegalEntity)
  (LegalEntity) ──FILED──► (Filing)
```

### Unique identifiers used

| Entity | Identifier | Standard |
|---|---|---|
| LegalEntity | `lei` | GLEIF LEI (20-char) |
| LegalEntity | `cik` | SEC CIK number |
| Instrument  | `figi` | OpenFIGI compositeFIGI |
| Instrument  | `ticker` | Exchange ticker symbol |
| Exchange    | `mic` | ISO 10383 MIC code |
| OntologyClass | `iri` | IRI / synthetic key |

> **Why multiple identifiers?** A company may appear in GLEIF with a LEI, in SEC data with a CIK, and in news text with just its name. Storing all identifiers on the same node enables disambiguation.

## 1.4 Neo4j Constraints & Indices

The file `_platform/schema/constraints.cypher` defines all constraints. They are applied automatically when any importer calls `self.ensure_constraints()`.

- **`UNIQUE` constraint** → `MERGE` finds the existing node instead of creating a duplicate.
- **`INDEX`** → fast lookups by name or CIK.

In [ ]:
# The actual constraints from _platform/schema/constraints.cypher
# (shown for reference; we'll apply them after connecting)

CONSTRAINTS = [
    # LegalEntity
    "CREATE CONSTRAINT fin_legal_entity_lei IF NOT EXISTS FOR (le:LegalEntity) REQUIRE le.lei IS UNIQUE",
    "CREATE INDEX fin_legal_entity_name     IF NOT EXISTS FOR (le:LegalEntity) ON (le.name)",
    "CREATE INDEX fin_legal_entity_cik      IF NOT EXISTS FOR (le:LegalEntity) ON (le.cik)",
    # Instrument
    "CREATE CONSTRAINT fin_instrument_figi  IF NOT EXISTS FOR (i:Instrument)   REQUIRE i.figi IS UNIQUE",
    "CREATE INDEX fin_instrument_ticker     IF NOT EXISTS FOR (i:Instrument)   ON (i.ticker)",
    # Exchange
    "CREATE CONSTRAINT fin_exchange_mic     IF NOT EXISTS FOR (ex:Exchange)    REQUIRE ex.mic IS UNIQUE",
    # OntologyClass
    "CREATE CONSTRAINT fin_ontology_iri     IF NOT EXISTS FOR (oc:OntologyClass) REQUIRE oc.iri IS UNIQUE",
]
print(f"Defined {len(CONSTRAINTS)} constraint statements.")

## 1.5 Connecting to Neo4j

In [ ]:
import os, sys
from pathlib import Path

REPO_ROOT = Path("../..").resolve()
sys.path.insert(0, str(REPO_ROOT))

NEO4J_URI      = os.getenv("NEO4J_URI",      "bolt://localhost:7687")
NEO4J_USER     = os.getenv("NEO4J_USER",     "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password")   # <-- update if needed
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")

print(f"Connecting to: {NEO4J_URI}  db={NEO4J_DATABASE}")

In [ ]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

def q(cypher: str, params: dict = None):
    """Run Cypher and return list of dicts."""
    with driver.session(database=NEO4J_DATABASE) as session:
        return [dict(r) for r in session.run(cypher, params or {})]

# Ping
try:
    print(q("RETURN 1 AS ping"))
    print("✓ Neo4j reachable.")
except Exception as e:
    print(f"✗ {e}")

In [ ]:
# Apply constraints (safe to re-run — IF NOT EXISTS)
with driver.session(database=NEO4J_DATABASE) as session:
    for stmt in CONSTRAINTS:
        try:
            session.run(stmt)
            print(f"  ✓ {stmt[7:40]}...")
        except Exception as e:
            print(f"  ~ already exists: {e}")
print("Schema ready.")

## 1.6 Creating Sample Data

The cells below insert a representative subset into Neo4j so you can explore the graph **without running the full importers**.

We create:
- 5 exchanges (XNAS, XNYS, XLON, XETR, XHKG)
- 10 well-known instruments
- 8 SIC sector classes
- 6 institutional holders (LegalEntity)
- 25 OWNS relationships

In [ ]:
# ── EXCHANGES ────────────────────────────────────────────────
SAMPLE_EXCHANGES = [
    {"mic": "XNAS", "name": "NASDAQ - ALL MARKETS",            "country": "US", "city": "New York",   "status": "UPDATED"},
    {"mic": "XNYS", "name": "NEW YORK STOCK EXCHANGE, INC.",   "country": "US", "city": "New York",   "status": "UPDATED"},
    {"mic": "XLON", "name": "LONDON STOCK EXCHANGE",           "country": "GB", "city": "London",    "status": "ACTIVE"},
    {"mic": "XETR", "name": "XETRA",                           "country": "DE", "city": "Frankfurt", "status": "ACTIVE"},
    {"mic": "XHKG", "name": "HONG KONG EXCHANGES AND CLEARING","country": "HK", "city": "Hong Kong", "status": "ACTIVE"},
]

# ISO 10383 STATUS NOTE:
#   ACTIVE  = newly published entry
#   UPDATED = previously active, since modified — but STILL CURRENTLY ACTIVE
#   EXPIRED/DELETED = no longer valid
#
# NASDAQ (XNAS) has status=UPDATED because it's been in use for decades.
# A filter for status='ACTIVE' only would MISS all major US exchanges!
# Fix: filter for status IN ('ACTIVE', 'UPDATED')

q("""
UNWIND $rows AS row
MERGE (ex:Exchange {mic: row.mic})
SET ex.name = row.name, ex.country = row.country,
    ex.city = row.city, ex.status  = row.status
""", {"rows": SAMPLE_EXCHANGES})

cnt = q("MATCH (ex:Exchange) RETURN count(ex) AS c")[0]["c"]
print(f"Exchange nodes: {cnt}")

In [ ]:
# ── INSTRUMENTS ────────────────────────────────────────────────
# figi values are real OpenFIGI compositeFIGIs
SAMPLE_INSTRUMENTS = [
    {"figi": "BBG000B9XRY4", "ticker": "AAPL",  "name": "APPLE INC",            "exchCode": "UW", "mic": "XNAS"},
    {"figi": "BBG000BPH459", "ticker": "MSFT",  "name": "MICROSOFT CORP",        "exchCode": "UW", "mic": "XNAS"},
    {"figi": "BBG009S39JX6", "ticker": "GOOGL", "name": "ALPHABET INC-CL A",     "exchCode": "UW", "mic": "XNAS"},
    {"figi": "BBG000BVPV84", "ticker": "AMZN",  "name": "AMAZON.COM INC",        "exchCode": "UW", "mic": "XNAS"},
    {"figi": "BBG000BBJQV0", "ticker": "NVDA",  "name": "NVIDIA CORP",           "exchCode": "UW", "mic": "XNAS"},
    {"figi": "BBG000N9MNX3", "ticker": "TSLA",  "name": "TESLA INC",             "exchCode": "UW", "mic": "XNAS"},
    {"figi": "BBG000DMBXR2", "ticker": "JPM",   "name": "JPMORGAN CHASE & CO",   "exchCode": "UN", "mic": "XNYS"},
    {"figi": "BBG000PSKYX7", "ticker": "V",     "name": "VISA INC-CLASS A SHARES","exchCode": "UN", "mic": "XNYS"},
    {"figi": "BBG000B9ZXB4", "ticker": "ABT",   "name": "ABBOTT LABORATORIES",   "exchCode": "UN", "mic": "XNYS"},
    {"figi": "BBG000GZQ728", "ticker": "XOM",   "name": "EXXON MOBIL CORP",      "exchCode": "UN", "mic": "XNYS"},
]

q("""
UNWIND $rows AS row
MERGE (i:Instrument {figi: row.figi})
SET i.ticker = row.ticker, i.name = row.name,
    i.exchCode = row.exchCode, i.mic = row.mic
""", {"rows": SAMPLE_INSTRUMENTS})

# LISTED_ON: link each instrument to its exchange by mic
q("MATCH (i:Instrument) MATCH (ex:Exchange {mic: i.mic}) MERGE (i)-[:LISTED_ON]->(ex)")

cnt_i = q("MATCH (i:Instrument) RETURN count(i) AS c")[0]["c"]
cnt_l = q("MATCH ()-[r:LISTED_ON]->() RETURN count(r) AS c")[0]["c"]
print(f"Instruments: {cnt_i},  LISTED_ON: {cnt_l}")

In [ ]:
# ── SIC SECTOR CLASSES (OntologyClass) ────────────────────────
SAMPLE_SIC = [
    {"iri": "SIC:7372", "label": "Prepackaged Software",            "sicCode": "7372", "sector": "Services"},
    {"iri": "SIC:3674", "label": "Semiconductors & Related Devices", "sicCode": "3674", "sector": "Manufacturing"},
    {"iri": "SIC:6022", "label": "National Commercial Banks",        "sicCode": "6022", "sector": "Finance, Insurance & Real Estate"},
    {"iri": "SIC:2911", "label": "Petroleum Refining",               "sicCode": "2911", "sector": "Manufacturing"},
    {"iri": "SIC:7389", "label": "Services-Business Services, NEC",  "sicCode": "7389", "sector": "Services"},
    {"iri": "SIC:5961", "label": "Catalog & Mail-Order Houses",      "sicCode": "5961", "sector": "Retail Trade"},
    {"iri": "SIC:8731", "label": "Commercial Physical & Bio Research","sicCode": "8731", "sector": "Services"},
    {"iri": "SIC:3559", "label": "Special Industry Machinery, NEC",  "sicCode": "3559", "sector": "Manufacturing"},
]

q("""
UNWIND $rows AS row
MERGE (oc:OntologyClass {iri: row.iri})
SET oc.label = row.label, oc.source = 'SIC',
    oc.sicCode = row.sicCode, oc.sector = row.sector
""", {"rows": SAMPLE_SIC})

# CLASSIFIED_AS: link each instrument to its SIC class
TICKER_SIC = [
    {"ticker": "AAPL",  "iri": "SIC:3674"},
    {"ticker": "MSFT",  "iri": "SIC:7372"},
    {"ticker": "GOOGL", "iri": "SIC:7389"},
    {"ticker": "AMZN",  "iri": "SIC:5961"},
    {"ticker": "NVDA",  "iri": "SIC:3674"},
    {"ticker": "TSLA",  "iri": "SIC:3559"},
    {"ticker": "JPM",   "iri": "SIC:6022"},
    {"ticker": "V",     "iri": "SIC:7389"},
    {"ticker": "ABT",   "iri": "SIC:8731"},
    {"ticker": "XOM",   "iri": "SIC:2911"},
]
q("""
UNWIND $rows AS row
MATCH (i:Instrument {ticker: row.ticker})
MATCH (oc:OntologyClass {iri: row.iri})
MERGE (i)-[:CLASSIFIED_AS]->(oc)
""", {"rows": TICKER_SIC})

cnt_oc = q("MATCH (oc:OntologyClass) RETURN count(oc) AS c")[0]["c"]
cnt_ca = q("MATCH ()-[r:CLASSIFIED_AS]->() RETURN count(r) AS c")[0]["c"]
print(f"OntologyClass: {cnt_oc},  CLASSIFIED_AS: {cnt_ca}")

In [ ]:
# ── INSTITUTIONAL HOLDERS + OWNS ──────────────────────────────
SAMPLE_HOLDERS = [
    {"lei": "LEI001VANGUARD00000001", "cik": "0000102909", "name": "The Vanguard Group, Inc.",       "jurisdiction": "US"},
    {"lei": "LEI002BLACKROCK0000001", "cik": "0001364742", "name": "BlackRock Inc.",                 "jurisdiction": "US"},
    {"lei": "LEI003FIDELITY0000001",  "cik": "0000315066", "name": "Fidelity Management & Research", "jurisdiction": "US"},
    {"lei": "LEI004STATESTR0000001",  "cik": "0000093751", "name": "State Street Corporation",       "jurisdiction": "US"},
    {"lei": "LEI005JPMORGAN0000001",  "cik": "0000019617", "name": "JPMorgan Asset Management",      "jurisdiction": "US"},
    {"lei": "LEI006GEODE000000001",   "cik": "0001595888", "name": "Geode Capital Management",       "jurisdiction": "US"},
]
q("""
UNWIND $rows AS row
MERGE (le:LegalEntity {lei: row.lei})
SET le.name = row.name, le.cik = row.cik, le.jurisdiction = row.jurisdiction
""", {"rows": SAMPLE_HOLDERS})

SAMPLE_OWNS = [
    # Vanguard: 5 holdings
    {"lei": "LEI001VANGUARD00000001", "ticker": "AAPL",  "shares": 1_350_000_000, "value_usd": 255_000_000_000},
    {"lei": "LEI001VANGUARD00000001", "ticker": "MSFT",  "shares": 975_000_000,   "value_usd": 378_000_000_000},
    {"lei": "LEI001VANGUARD00000001", "ticker": "AMZN",  "shares": 730_000_000,   "value_usd": 130_000_000_000},
    {"lei": "LEI001VANGUARD00000001", "ticker": "NVDA",  "shares": 820_000_000,   "value_usd": 990_000_000_000},
    {"lei": "LEI001VANGUARD00000001", "ticker": "XOM",   "shares": 210_000_000,   "value_usd": 25_000_000_000},
    # BlackRock: 5 holdings
    {"lei": "LEI002BLACKROCK0000001", "ticker": "AAPL",  "shares": 1_050_000_000, "value_usd": 198_000_000_000},
    {"lei": "LEI002BLACKROCK0000001", "ticker": "MSFT",  "shares": 780_000_000,   "value_usd": 302_000_000_000},
    {"lei": "LEI002BLACKROCK0000001", "ticker": "JPM",   "shares": 220_000_000,   "value_usd":  44_000_000_000},
    {"lei": "LEI002BLACKROCK0000001", "ticker": "V",     "shares": 130_000_000,   "value_usd":  38_000_000_000},
    {"lei": "LEI002BLACKROCK0000001", "ticker": "XOM",   "shares": 185_000_000,   "value_usd":  22_000_000_000},
    # Fidelity: 4 holdings
    {"lei": "LEI003FIDELITY0000001",  "ticker": "AAPL",  "shares": 410_000_000,   "value_usd":  77_000_000_000},
    {"lei": "LEI003FIDELITY0000001",  "ticker": "NVDA",  "shares": 290_000_000,   "value_usd": 350_000_000_000},
    {"lei": "LEI003FIDELITY0000001",  "ticker": "TSLA",  "shares": 165_000_000,   "value_usd":  54_000_000_000},
    {"lei": "LEI003FIDELITY0000001",  "ticker": "ABT",   "shares":  82_000_000,   "value_usd":   8_600_000_000},
    # State Street: 4 holdings
    {"lei": "LEI004STATESTR0000001",  "ticker": "MSFT",  "shares": 460_000_000,   "value_usd": 178_000_000_000},
    {"lei": "LEI004STATESTR0000001",  "ticker": "AMZN",  "shares": 280_000_000,   "value_usd":  50_000_000_000},
    {"lei": "LEI004STATESTR0000001",  "ticker": "JPM",   "shares": 158_000_000,   "value_usd":  31_600_000_000},
    {"lei": "LEI004STATESTR0000001",  "ticker": "XOM",   "shares": 140_000_000,   "value_usd":  16_700_000_000},
    # JPMorgan AM: 3 holdings
    {"lei": "LEI005JPMORGAN0000001",  "ticker": "AAPL",  "shares": 195_000_000,   "value_usd":  36_800_000_000},
    {"lei": "LEI005JPMORGAN0000001",  "ticker": "GOOGL", "shares":  98_000_000,   "value_usd":  17_600_000_000},
    {"lei": "LEI005JPMORGAN0000001",  "ticker": "V",     "shares":  65_000_000,   "value_usd":  19_000_000_000},
    # Geode: 4 holdings
    {"lei": "LEI006GEODE000000001",   "ticker": "AAPL",  "shares": 320_000_000,   "value_usd":  60_400_000_000},
    {"lei": "LEI006GEODE000000001",   "ticker": "NVDA",  "shares": 210_000_000,   "value_usd": 253_500_000_000},
    {"lei": "LEI006GEODE000000001",   "ticker": "TSLA",  "shares":  95_000_000,   "value_usd":  31_000_000_000},
    {"lei": "LEI006GEODE000000001",   "ticker": "GOOGL", "shares":  70_000_000,   "value_usd":  12_600_000_000},
]
q("""
UNWIND $rows AS row
MATCH (le:LegalEntity {lei: row.lei})
MATCH (i:Instrument {ticker: row.ticker})
MERGE (le)-[r:OWNS]->(i)
SET r.shares = row.shares, r.valueUsd = row.value_usd, r.asOf = '2024-12-31'
""", {"rows": SAMPLE_OWNS})

cnt_le = q("MATCH (le:LegalEntity) RETURN count(le) AS c")[0]["c"]
cnt_ow = q("MATCH ()-[r:OWNS]->() RETURN count(r) AS c")[0]["c"]
print(f"LegalEntity: {cnt_le},  OWNS: {cnt_ow}")

In [ ]:
# Graph inventory
print("=== Node counts ===")
for label in ["Exchange", "Instrument", "LegalEntity", "OntologyClass"]:
    cnt = q(f"MATCH (n:{label}) RETURN count(n) AS c")[0]["c"]
    print(f"  ({label}): {cnt}")

print("\n=== Relationship counts ===")
for rel in ["LISTED_ON", "CLASSIFIED_AS", "OWNS"]:
    cnt = q(f"MATCH ()-[r:{rel}]->() RETURN count(r) AS c")[0]["c"]
    print(f"  [:{rel}]: {cnt}")

---
# PART 2 — ch04_fin: Building the Financial Graph

## 2.1 Architecture Overview

ch04_fin pulls from four real-world data sources:

```
ISO 10383 MIC CSV ──► import_exchanges.py ──► (:Exchange) nodes
                                                      │ LISTED_ON
OpenFIGI API ──────► import_instruments_figi.py ──► (:Instrument) nodes
                                                      │ CLASSIFIED_AS
SEC EDGAR JSON ────► import_sector_mapping.py ──► (:OntologyClass) nodes
                                                      │
SEC EDGAR 13F ─────► import_ownership_13f.py ──► (:LegalEntity)-[:OWNS]→

Analysis:
  community_louvain.py  ──► communityLouvain property on LegalEntity
  centrality_pagerank.py ──► pagerank property on LegalEntity
```

Each step is **idempotent** — uses `MERGE`, never creates duplicates.

## 2.2 Step 1 — Exchange Nodes (ISO 10383 MIC)

### What is a MIC?

ISO 10383 assigns 4-letter **Market Identifier Codes** to every regulated trading venue:

| MIC | Exchange | Country |
|---|---|---|
| XNAS | NASDAQ — All Markets | US |
| XNYS | New York Stock Exchange | US |
| XLON | London Stock Exchange | GB |
| XTKS | Tokyo Stock Exchange | JP |
| XHKG | Hong Kong Exchanges | HK |

### The ACTIVE vs UPDATED status issue (real bug fixed in this project)

ISO 10383 uses three statuses:
- `ACTIVE` — newly published entry
- `UPDATED` — previously active, since modified (**but still active**)
- `EXPIRED` / `DELETED` — no longer valid

All major US venues (XNAS, XNGS, XNMS, XNCM) have `status = UPDATED`.  
The original code filtered `status == 'ACTIVE'` only → NASDAQ was missing → 18 instruments had no LISTED_ON edge.

**Fix:** `status IN ('ACTIVE', 'UPDATED')`

In [ ]:
# The Cypher used by import_exchanges.py
MERGE_EXCHANGE = """
UNWIND $batch AS row
MERGE (ex:Exchange {mic: row.mic})
SET ex.name         = row.name,
    ex.country      = row.country,
    ex.operatingMic = row.operatingMic,
    ex.city         = row.city,
    ex.status       = row.status,
    ex.runId        = row.runId,
    ex.ingestedAt   = row.ingestedAt
"""
print("Exchange merge template ready.")

# Verify sample data
print("\nSample exchanges in graph:")
for r in q("MATCH (ex:Exchange) RETURN ex.mic AS mic, ex.country AS c, ex.name AS name ORDER BY ex.name"):
    print(f"  {r['mic']}  ({r['c']})  {r['name']}")

## 2.3 Step 2 — Instrument Nodes (OpenFIGI)

### What is a FIGI?

**Financial Instrument Global Identifier** — an open standard for securities (like ISBN for books).

- **compositeFIGI** — identifies the security globally (same AAPL FIGI regardless of exchange)
- **shareFIGI** — specific to one listing on one exchange

We use `compositeFIGI` as the node key → one `Instrument` node per company.

### 3-pass probing strategy

Without an exchange hint, OpenFIGI might return the Euronext Paris ADR of AAPL instead of NASDAQ:

```
Pass 1: exchCode=UW  → NASDAQ Global Select (AAPL, MSFT, NVDA …)
Pass 2: exchCode=UN  → NYSE (JPM, V, XOM …)
Pass 3: no filter   → catches anything remaining (BRK.B etc.)
```

Each pass only re-queries tickers not yet resolved — no wasted API calls.

### exchCode → MIC mapping

OpenFIGI returns short `exchCode` values, not MIC codes:

```python
EXCHCODE_TO_MIC = {
    "UN": "XNYS",   # NYSE
    "UW": "XNAS",   # NASDAQ Global Select
    "UQ": "XNAS",   # NASDAQ Global Market
    "UR": "XNCM",   # NASDAQ Capital Market
    "UP": "ARCX",   # NYSE Arca
    "LN": "XLON",   # London Stock Exchange
    "GY": "XETR",   # XETRA (Frankfurt)
    ...
}
```

In [ ]:
# Verify sample instruments
print("Sample instruments:")
print(f"  {'TICKER':<8} {'FIGI':<15} {'EXCHCODE':<9} {'MIC':<6} NAME")
print("  " + "-"*75)
for r in q("MATCH (i:Instrument) RETURN i.ticker AS t, i.figi AS f, i.exchCode AS e, i.mic AS m, i.name AS n ORDER BY i.ticker"):
    print(f"  {r['t']:<8} {r['f']:<15} {r['e']:<9} {r['m']:<6} {r['n']}")

## 2.4 Step 3 — LISTED_ON Relationships

```cypher
UNWIND $batch AS row
MATCH (i:Instrument {figi: row.figi})
MATCH (ex:Exchange {mic: row.mic})
MERGE (i)-[:LISTED_ON]->(ex)
```

**Why MERGE and not CREATE?**  
`MERGE` is idempotent — if the relationship already exists it returns the existing one. Re-running the importer is safe.

In [ ]:
# Instruments and their exchanges
print("Instruments per exchange:")
for r in q("""
    MATCH (i:Instrument)-[:LISTED_ON]->(ex:Exchange)
    RETURN ex.mic AS mic, ex.name AS exchange, count(i) AS n, collect(i.ticker) AS tickers
    ORDER BY n DESC
"""):
    print(f"  {r['mic']} ({r['exchange']}): {r['n']} instruments → {sorted(r['tickers'])}")

## 2.5 Step 4 — Sector Mapping (SEC SIC codes)

### What is SIC?

**Standard Industrial Classification** — a 4-digit code the SEC requires on all filings.

```
SIC 7372 → Prepackaged Software          (MSFT, GOOGL)
SIC 3674 → Semiconductors                (NVDA, AAPL)
SIC 6022 → National Commercial Banks     (JPM)
SIC 2911 → Petroleum Refining            (XOM)
```

### How the importer works

1. `GET data.sec.gov/files/company_tickers_exchange.json` → ticker → CIK
2. `GET data.sec.gov/submissions/CIK{cik}.json` → `sic`, `sicDescription`
3. `MERGE (:OntologyClass {iri: 'SIC:XXXX'})`
4. `MERGE (i:Instrument)-[:CLASSIFIED_AS]->(oc:OntologyClass)`

In [ ]:
# Sector distribution
print("Instruments by sector:")
for r in q("""
    MATCH (i:Instrument)-[:CLASSIFIED_AS]->(oc:OntologyClass)
    RETURN oc.sector AS sector, oc.label AS label, collect(i.ticker) AS tickers, count(i) AS n
    ORDER BY n DESC, sector
"""):
    print(f"  {r['sector']:<40} {r['label']:<40} {sorted(r['tickers'])}")

## 2.6 Step 5 — Ownership Graph (SEC 13F filings)

### What is a 13F?

Any institutional manager with ≥ $100M AUM must file **Form 13F** quarterly with the SEC, disclosing every equity holding ≥ $200K or 10,000 shares.

This creates a public, machine-readable ownership graph updated 4× per year.

### How the importer works

1. For each instrument name (e.g. `"APPLE INC"`), search SEC EDGAR EFTS for 13F-HR filings.
2. Extract `cik` + `display_name` of each filer.
3. `MERGE (:LegalEntity {cik: row.cik})`
4. `MERGE (le)-[:OWNS]->(i:Instrument {ticker: row.ticker})`

In [ ]:
# Who holds the most instruments?
print("Institutional holders by # of holdings:")
for r in q("""
    MATCH (h:LegalEntity)-[r:OWNS]->(i:Instrument)
    RETURN h.name AS fund, count(i) AS n, sum(r.valueUsd) AS aum
    ORDER BY n DESC
"""):
    aum = f"${r['aum']/1e9:.1f}B" if r["aum"] else "N/A"
    print(f"  {r['fund']:<45} {r['n']:>3} holdings  {aum}")

In [ ]:
# Most widely held instruments
print("Most widely held instruments:")
for r in q("""
    MATCH (h:LegalEntity)-[:OWNS]->(i:Instrument)
    RETURN i.ticker AS ticker, i.name AS name, count(h) AS holders
    ORDER BY holders DESC
"""):
    bar = "█" * r["holders"]
    print(f"  {r['ticker']:<6} {r['name']:<35} {bar} ({r['holders']})")

---
# PART 3 — Graph Exploration with Cypher

## 3.1 Cypher Quick Reference

```cypher
(n)                -- any node
(n:Label)          -- node with label
(n {prop: val})    -- node with property filter
-[:REL]->          -- directed relationship
<-[:REL]-          -- reverse direction
-[*1..3]->         -- variable-length 1–3 hops
MATCH … WHERE …    -- pattern + filter
RETURN …           -- project results
ORDER BY … LIMIT … -- sort and paginate
```

In [ ]:
# Q: Find all instruments listed on NASDAQ
print("Instruments on NASDAQ (XNAS):")
for r in q("""
    MATCH (i:Instrument)-[:LISTED_ON]->(ex:Exchange {mic: 'XNAS'})
    RETURN i.ticker AS t, i.name AS n ORDER BY t
"""):
    print(f"  {r['t']:<8} {r['n']}")

In [ ]:
# Q: Full instrument context — exchange + sector
print(f"  {'TICKER':<8} {'MIC':<6} {'SECTOR':<35} SIC LABEL")
print("  " + "-"*85)
for r in q("""
    MATCH (i:Instrument)-[:LISTED_ON]->(ex:Exchange)
    OPTIONAL MATCH (i)-[:CLASSIFIED_AS]->(oc:OntologyClass)
    RETURN i.ticker AS t, ex.mic AS m,
           coalesce(oc.sector,'—') AS sec, coalesce(oc.label,'—') AS sic
    ORDER BY i.ticker
"""):
    print(f"  {r['t']:<8} {r['m']:<6} {r['sec']:<35} {r['sic']}")

## 3.2 Filtering and Aggregation

In [ ]:
# Q: Portfolio composition by sector and exchange
print("Holdings by fund and sector:")
for r in q("""
    MATCH (h:LegalEntity)-[r:OWNS]->(i:Instrument)-[:CLASSIFIED_AS]->(oc:OntologyClass)
    RETURN h.name AS fund, oc.sector AS sector,
           collect(i.ticker) AS tickers, count(i) AS n
    ORDER BY fund, n DESC
"""):
    print(f"  {r['fund']:<45} {r['sector']:<35} {sorted(r['tickers'])}")

In [ ]:
# Q: Portfolio value per institution
print(f"  {'FUND':<45} {'POSITIONS':>9}  {'AUM':>12}")
print("  " + "-"*70)
for r in q("""
    MATCH (h:LegalEntity)-[r:OWNS]->(i:Instrument)
    WHERE r.valueUsd IS NOT NULL
    RETURN h.name AS fund, count(i) AS pos, sum(r.valueUsd) AS aum
    ORDER BY aum DESC
"""):
    print(f"  {r['fund']:<45} {r['pos']:>9}  ${r['aum']/1e9:>10.1f}B")

## 3.3 Shortest Paths

In [ ]:
# Q: What connects BlackRock to XOM?
result = q("""
    MATCH path = shortestPath(
        (br:LegalEntity {name: 'BlackRock Inc.'})-[*..6]-(xom:Instrument {ticker: 'XOM'})
    )
    RETURN [n IN nodes(path) |
        CASE WHEN n:LegalEntity  THEN n.name
             WHEN n:Instrument   THEN n.ticker
             WHEN n:Exchange     THEN n.mic
             WHEN n:OntologyClass THEN n.label
             ELSE 'node' END
    ] AS labels, length(path) AS hops
""")
if result:
    print(f"Shortest path ({result[0]['hops']} hops):")
    print("  " + " → ".join(result[0]["labels"]))
else:
    print("No path found.")

In [ ]:
# Q: Portfolio overlap between Vanguard and State Street
result = q("""
    MATCH (a:LegalEntity {name: 'The Vanguard Group, Inc.'})
          -[:OWNS]->(i:Instrument)
          <-[:OWNS]-(b:LegalEntity {name: 'State Street Corporation'})
    RETURN collect(i.ticker) AS shared, count(i) AS overlap
""")
print(f"Shared holdings: {result[0]['overlap']}")
print(f"Tickers: {sorted(result[0]['shared'])}")

## 3.4 Community Detection (Louvain / Connected Components)

**Community detection** groups nodes that are densely connected to each other.  
In a financial ownership graph, communities often map to:
- Funds with similar portfolios (index fund cluster)
- Corporate ownership trees (parent-subsidiary)
- Sector-focused investors

### Two modes in `community_louvain.py`

| Mode | Requirement | Algorithm |
|---|---|---|
| GDS Louvain | Neo4j GDS plugin installed | `CALL gds.louvain.write(...)` — true Louvain |
| Cypher fallback | Community Edition (no GDS) | Iterative label propagation (connected components) |

**Louvain** maximises *modularity* Q:

$$Q = \frac{1}{2m} \sum_{ij} \left[ A_{ij} - \frac{k_i k_j}{2m} \right] \delta(c_i, c_j)$$

where $A$ is the adjacency matrix, $k_i$ is degree of node $i$, $m$ is number of edges, and $\delta(c_i, c_j)=1$ if nodes are in the same community.

In [ ]:
# Detect GDS availability
def has_gds():
    try:
        q("RETURN gds.version() AS v")
        return True
    except Exception:
        return False

gds = has_gds()
print(f"GDS available: {gds}")

In [ ]:
if gds:
    # ── GDS Louvain ─────────────────────────────────────────────
    print("Running GDS Louvain...")
    try:
        q("CALL gds.graph.drop('fin_tut', false)")
    except Exception:
        pass

    # Project: LegalEntity nodes + OWNS relationships (undirected)
    # (we treat two holders as "related" if they own the same instrument)
    q("""
        CALL gds.graph.project(
            'fin_tut',
            'LegalEntity',
            { OWNS: { orientation: 'UNDIRECTED' } }
        )
    """)

    r = q("""
        CALL gds.louvain.write('fin_tut', { writeProperty: 'communityLouvain' })
        YIELD communityCount, modularity, ranLevels
        RETURN communityCount, modularity, ranLevels
    """)
    print(f"Communities: {r[0]['communityCount']}, Modularity: {r[0]['modularity']:.4f}, Levels: {r[0]['ranLevels']}")
    q("CALL gds.graph.drop('fin_tut', false)")

else:
    # ── Cypher connected-component approximation ─────────────────
    # Intuition: two holders that share an instrument are "neighbours".
    # We propagate the minimum node ID through shared-instrument edges
    # until convergence → nodes in the same component get the same ID.
    print("Running Cypher connected-component approximation...")

    q("MATCH (le:LegalEntity) SET le.communityLouvain = id(le)")

    for i in range(10):
        r = q("""
            MATCH (a:LegalEntity)-[:OWNS]->(i:Instrument)<-[:OWNS]-(b:LegalEntity)
            WHERE a.communityLouvain > b.communityLouvain
            SET a.communityLouvain = b.communityLouvain
            RETURN count(a) AS updated
        """)
        cnt = r[0]["updated"]
        print(f"  Iteration {i+1}: updated {cnt} nodes.")
        if cnt == 0:
            print(f"  Converged.")
            break

print("communityLouvain property written.")

In [ ]:
# Inspect communities
print("Community summary:")
for r in q("""
    MATCH (le:LegalEntity)
    WHERE le.communityLouvain IS NOT NULL
    RETURN le.communityLouvain AS cid, count(le) AS members,
           collect(le.name)[..3] AS names
    ORDER BY members DESC LIMIT 10
"""):
    names = ", ".join(r["names"])
    print(f"  Community {r['cid']}: {r['members']} member(s) → {names}")

# Explanation
print("""
Interpretation:
  All 6 sample holders share at least one instrument (e.g. AAPL),
  so the connected-component algorithm merges them into ONE community.

  With the full 13F data (210+ holders), you would see:
    - A few large communities (major index funds with overlapping mega-cap holdings)
    - Many singleton communities (niche funds with unique portfolios)
""")

## 3.5 Centrality — PageRank

**PageRank** asks: *"Which nodes are most important based on who links to them?"*

In the financial graph:
- High PageRank `LegalEntity` → owned or linked by many other important entities
- Systemic importance indicator

### Algorithm

$$PR(u) = \frac{1-d}{N} + d \sum_{v \in B_u} \frac{PR(v)}{L(v)}$$

- $d = 0.85$ — damping factor (85% chance of following a link)
- $N$ — total node count
- $B_u$ — nodes pointing to $u$
- $L(v)$ — out-degree of $v$

In [ ]:
if gds:
    print("Running GDS PageRank...")
    try:
        q("CALL gds.graph.drop('fin_pr', false)")
    except Exception:
        pass
    q("""
        CALL gds.graph.project('fin_pr', 'LegalEntity',
            { OWNS: { orientation: 'NATURAL' } })
    """)
    r = q("""
        CALL gds.pageRank.write('fin_pr', {
            writeProperty: 'pagerank', maxIterations: 20, dampingFactor: 0.85
        })
        YIELD nodePropertiesWritten, ranIterations, didConverge
        RETURN nodePropertiesWritten, ranIterations, didConverge
    """)
    print(f"Written: {r[0]['nodePropertiesWritten']}, converged: {r[0]['didConverge']}")
    q("CALL gds.graph.drop('fin_pr', false)")

else:
    # ── Cypher power-iteration PageRank ─────────────────────────
    damping, iterations = 0.85, 10
    total = q("MATCH (le:LegalEntity) RETURN count(le) AS c")[0]["c"]
    init  = 1.0 / total if total > 0 else 1.0

    q("MATCH (le:LegalEntity) SET le.pagerank = $init", {"init": init})

    for i in range(iterations):
        q("""
            MATCH (le:LegalEntity)
            OPTIONAL MATCH (src:LegalEntity)-[:OWNS|PARENT_OF]->(le)
            WITH le,
                 sum(CASE WHEN src IS NOT NULL AND src.pagerank > 0
                          THEN src.pagerank /
                               toFloat(size([(src)-[:OWNS|PARENT_OF]->() | 1]))
                          ELSE 0 END) AS incoming
            SET le.pagerank = $base + $d * incoming
        """, {"base": (1.0 - damping) / total, "d": damping})
        print(f"  Iteration {i+1}/{iterations}")

    print("PageRank complete.")

In [ ]:
# Top LegalEntity nodes by PageRank
print("Top 10 by PageRank:")
for rank, r in enumerate(q("""
    MATCH (le:LegalEntity)
    WHERE le.pagerank IS NOT NULL
    RETURN le.name AS name, le.pagerank AS pr
    ORDER BY pr DESC LIMIT 10
"""), 1):
    bar = "█" * min(20, max(1, int(r["pr"] * 10000 * 20)))
    print(f"  #{rank:<2} {r['name']:<45} {r['pr']:.6f}  {bar}")

## 3.6 Advanced Pattern: Portfolio Risk Analysis

In [ ]:
# Q: Sector concentration — who is over-concentrated?
print("Sector concentration (>30% of portfolio in one sector):")
print(f"  {'FUND':<45} {'SECTOR':<35} {'POS':>4}  {'%':>5}")
print("  " + "-"*95)
for r in q("""
    MATCH (h:LegalEntity)-[:OWNS]->(i:Instrument)
    OPTIONAL MATCH (i)-[:CLASSIFIED_AS]->(oc:OntologyClass)
    WITH h, count(i) AS total, coalesce(oc.sector,'Other') AS sector, count(i) AS n
    WITH h, total, sector, n,
         toFloat(n) / total AS concentration
    WHERE concentration > 0.30
    RETURN h.name AS fund, sector, n AS positions,
           total AS total_pos, round(concentration*100) AS pct
    ORDER BY pct DESC
"""):
    print(f"  {r['fund']:<45} {r['sector']:<35} {r['positions']:>2}/{r['total_pos']:<2}  {r['pct']:>5.0f}%")

In [ ]:
# Q: For AAPL — who owns it, on which exchange, and what sector?
r = q("""
    MATCH (i:Instrument {ticker: 'AAPL'})
    OPTIONAL MATCH (i)-[:LISTED_ON]->(ex:Exchange)
    OPTIONAL MATCH (i)-[:CLASSIFIED_AS]->(oc:OntologyClass)
    OPTIONAL MATCH (h:LegalEntity)-[:OWNS]->(i)
    RETURN i.name AS name, i.figi AS figi,
           ex.mic AS exchange, oc.label AS sector,
           collect(h.name) AS holders
""")
if r:
    row = r[0]
    print(f"Instrument: {row['name']}")
    print(f"  FIGI:     {row['figi']}")
    print(f"  Exchange: {row['exchange']}")
    print(f"  Sector:   {row['sector']}")
    print(f"  Holders ({len(row['holders'])}):")
    for h in sorted(row["holders"]):
        print(f"    - {h}")

## 3.7 Summary

| Concept | Key takeaway |
|---|---|
| Knowledge graph data model | Nodes + relationships; heterogeneous sources linked naturally |
| `MERGE` semantics | Finds existing node/rel by unique key; safe to re-run importers |
| ISO 10383 MIC codes | 4-letter exchange IDs; `ACTIVE + UPDATED` = currently active |
| OpenFIGI compositeFIGI | Stable per-security key; 3-pass exchCode probing for correct MIC |
| SEC SIC codes | 4-digit industry classifier; fetched from EDGAR submissions API |
| SEC 13F filings | Quarterly ownership disclosures → OWNS edges |
| Community detection | Louvain (GDS) or label-propagation (Cypher) identifies ownership clusters |
| PageRank | Power-iteration identifies systemically important nodes |
| `FinImporterBase` | Auto-injects `runId` + `ingestedAt`; `batch_store()` for bulk upserts |

### Neo4j Browser quick-start

Paste into `http://localhost:7474`:

```cypher
// See the entire sample graph
MATCH (n) RETURN n LIMIT 100

// Ownership subgraph around AAPL
MATCH path = (h:LegalEntity)-[:OWNS]->(i:Instrument {ticker:'AAPL'})
             -[:LISTED_ON]->(ex:Exchange)
RETURN path

// Full chain: holder → instrument → sector
MATCH path = (h:LegalEntity)-[:OWNS]->(i:Instrument)
             -[:CLASSIFIED_AS]->(oc:OntologyClass)
RETURN path LIMIT 50
```

### Next chapters
- **ch03_fin** — GLEIF legal entities + FIBO ontology
- **ch05_fin** — NER on financial text → Mention nodes
- **ch08_fin** — RAG over the knowledge graph

In [ ]:
# Final summary
print("=" * 60)
print("  FINAL GRAPH SUMMARY")
print("=" * 60)
for label in ["Exchange", "Instrument", "LegalEntity", "OntologyClass"]:
    cnt = q(f"MATCH (n:{label}) RETURN count(n) AS c")[0]["c"]
    print(f"  ({label}): {cnt}")
print()
for rel in ["LISTED_ON", "CLASSIFIED_AS", "OWNS"]:
    cnt = q(f"MATCH ()-[r:{rel}]->() RETURN count(r) AS c")[0]["c"]
    print(f"  [:{rel}]: {cnt}")
pr = q("MATCH (le:LegalEntity) WHERE le.pagerank IS NOT NULL RETURN count(le) AS c")[0]["c"]
cm = q("MATCH (le:LegalEntity) WHERE le.communityLouvain IS NOT NULL RETURN count(le) AS c")[0]["c"]
print(f"  Nodes with pagerank:         {pr}")
print(f"  Nodes with communityLouvain: {cm}")
print("=" * 60)

In [ ]:
driver.close()
print("Neo4j driver closed.")